# 0.0 Bibliotecas Importadas

In [85]:
import pandas as pd
import numpy as np
import sys

from sklearn.model_selection import train_test_split as ms

# 0.1 Carregando Dataset

In [64]:
df_raw = pd.read_csv('/mnt/c/Users/T.i/Documents/Repos/projeto_RH/dataset/HR-Employee-Attrition-balanced.csv')

# 0.2 Leitura do Dataset

# 0.3 Separação dos Dados: train, test, split

In [65]:
#separação dos dados, definindo quem é x e quem é y. 
#no x a gente tira a coluna target(alvo nosso) e no y escolhe exatamente apenas ela

x = df_raw.drop('Attrition', axis=1)
y = df_raw['Attrition']



In [66]:
#escrevendo o código agora para separar o dataset em dois: train e test
#só que depois vamos juntar x e y em um df2, pra voltar a ser um dataset completo
#pra gente trabalhar nele ainda


x_train, x_test, y_train, y_test = ms(x, y, stratify=y, test_size=0.25, random_state=42 )

In [67]:
#conferindo se ta tudo certo com dataset x
print(x.shape)
x.head(5)

(1761, 34)


,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,58,Travel_Rarely,601,Research & Development,7,4,Medical,1,1360,3,...,4,80,0,31,0,2,10,9,5,9
1,27,Travel_Frequently,1341,Human Resources,21,2,Human Resources,1,1950,0,...,1,80,0,0,2,3,0,0,0,0
2,50,Travel_Rarely,407,Sales,28,3,Marketing,1,2049,3,...,2,80,1,19,3,2,2,2,1,0
3,39,Non-Travel,439,Research & Development,9,3,Life Sciences,1,1132,3,...,3,80,0,9,2,2,5,4,0,3
4,32,Travel_Rarely,601,Sales,7,5,Marketing,1,1446,4,...,3,80,1,7,3,2,4,3,0,3


In [68]:
#conferindo se ta tudo certo com dataset y
print(y.shape)
y.head(5)

(1761,)


0    Yes
1    Yes
2    Yes
3     No
4     No
Name: Attrition, dtype: str

In [69]:
#escrevendo o código agora para separar o dataset em dois: train e test
#só que depois vamos juntar x e y em um df2, pra voltar a ser um dataset completo
#pra gente trabalhar nele ainda


x_train, x_test, y_train, y_test = ms(x, y, stratify=y, test_size=0.25, random_state=42 )

In [70]:
#conferindo os dados separados
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(1320, 34)
(1320,)
(441, 34)
(441,)


# 1.0 Entrega Final do Projeto

In [71]:
baseline_df = df_raw.copy()

baseline_df['Annual_Salary'] = baseline_df['MonthlyIncome']*12
baseline_df['Replacement_Multiplier'] = baseline_df['JobLevel'].apply(lambda x: 0.40 if x <= 2 else (1.00 if x == 3 else 2.00))
baseline_df['Replacement_Cost'] = baseline_df['Annual_Salary'] * baseline_df['Replacement_Multiplier']

#formando um df apenas os funcionários que tiveram a saída da empresa
leaving_df = baseline_df.loc[baseline_df['Attrition'] == 'Yes', :]

#metricas
total_employees = len(baseline_df)
employees_leaving = len(leaving_df)
attrition_rate = (employees_leaving / total_employees) * 100
avg_replacement_cost = leaving_df['Replacement_Cost'].mean()
total_annual_cost = leaving_df['Replacement_Cost'].sum()

print("="*60)
print("SITUAÇÃO ATUAL (BASELINE SEM MACHINE LEARNING)")
print("="*60)
print(f"Total de Funcionários: {total_employees}")
print(f"Funcionários que saem/ano (Rotatividade): {employees_leaving}")
print(f"Taxa de Rotatividade Anual: {attrition_rate:.2f}%")

print("-"*60)
print(f"Custo médio de substituição/pessoa: ${avg_replacement_cost:,.2f}")
print(f"Custo total anual de rotatividade: ${total_annual_cost:,.2f}")
print("="*60)

SITUAÇÃO ATUAL (BASELINE SEM MACHINE LEARNING)
Total de Funcionários: 1761
Funcionários que saem/ano (Rotatividade): 528
Taxa de Rotatividade Anual: 29.98%
------------------------------------------------------------
Custo médio de substituição/pessoa: $46,195.30
Custo total anual de rotatividade: $24,391,116.00


### 1.1 Simulação do ROI - Simples

In [72]:
#o quanto o meu modelo performa
recall = 0.10

#o quanto o time de RH performa para trabalhar em cima dos empregados que o modelo acusou
retention_rate = 0.30

#custos:

#manutenção
maintenance_annual_cost = 2000

#custo RH para intervir
invtervetion_cost = 5000

#custo de produzir o modelo em si
production_cost = 20000

#pessoas que o algoritmo aponta como potencial de demissão
true_positive = int(employees_leaving * recall)

#desempenho do rh
retention_employee = int(true_positive * retention_rate)
avg_saving = retention_employee * avg_replacement_cost

#custo anual
annual_cost = maintenance_annual_cost + invtervetion_cost + production_cost

#resultado liquido
liquid_savings = avg_saving - annual_cost
roi = (avg_saving / annual_cost) * 100

print(f'A média de economia da retenção dos empregados foi de: R${avg_saving:.2f}')
print(f'O ROI foi de: R${roi:.2f}')




A média de economia da retenção dos empregados foi de: R$692929.43
O ROI foi de: R$2566.41


### 1.2 Simulação do ROI - Elaborado

In [73]:
def simular_cenario_roi(recall, precision, taxa_retencao):

    # Constantes do Report
    CUSTO_MANUTENCAO_ANUAL = 20000
    CUSTO_INTERVENCAO = 5000

    # 1. Pessoas que a Máquina Aponta (Recall)
    verdadeiros_positivos = int(employees_leaving * recall)

    # 2. Pessoas Retidas de Fato pelo RH
    funcionarios_retidos = int(verdadeiros_positivos * taxa_retencao)

    # 3. Benefício Bruto (Pessoas salvas x O quanto a empresa gastaria pra demiti-las)
    economia_bruta = funcionarios_retidos * avg_replacement_cost
    
    # 5. Custos do Processo (Precisamos pagar pela manutenção de TI e pelas reuniões do RH)
    # Quantos alarmes o modelo gerou no total para conseguirmos os Verdadeiros Positivos?
    # Se a precisão é 70%, o volume total alardado é Verdadeiros Positivos / 0.70
    total_intencoes_rh = verdadeiros_positivos / precision if precision > 0 else 0
    custo_intervencoes = total_intencoes_rh * CUSTO_INTERVENCAO
    
    custos_anuais_totais = CUSTO_MANUTENCAO_ANUAL + custo_intervencoes
    
    # 6. Balanço Final
    economia_liquida = economia_bruta - custos_anuais_totais
    roi_anual = (economia_liquida / custos_anuais_totais) * 100 if custos_anuais_totais > 0 else 0
    
    return {
        "Cenário": "Simulado",
        "Precision": f"{precision*100:.0f}%",
        "Recall": f"{recall*100:.0f}%",
        "Retenção": f"{taxa_retencao*100:.0f}%",
        "Func. Retidos": funcionarios_retidos,
        "Economia Líquida (Anual)": f"${economia_liquida:,.0f}",
        "ROI Anual (%)": f"{roi_anual:.0f}%"
    }

# Criando tabela interativa com cenários conservador, moderado e otimista
tabela_analise = [
    simular_cenario_roi(precision = 0.60, recall=0.50, taxa_retencao=0.20),
    simular_cenario_roi(precision = 0.70, recall=0.60, taxa_retencao=0.30),
    simular_cenario_roi(precision = 0.80, recall=0.70, taxa_retencao=0.40),
]

df_potencial_roi = pd.DataFrame(tabela_analise)

# Renomear índices para identificar os cenários
df_potencial_roi['Cenário'] = ['Conservador', 'Moderado', 'Otimista']

display(df_potencial_roi)

,Cenário,Precision,Recall,Retenção,Func. Retidos,Economia Líquida (Anual),ROI Anual (%)
0,Conservador,60%,50%,20%,52,"$182,155",8%
1,Moderado,70%,60%,30%,94,"$2,065,215",91%
2,Otimista,80%,70%,40%,147,"$4,464,458",192%


# 2.0 Qualidade de Dados

In [74]:
df1 = df_raw.copy()

In [75]:
#dimensão dos dados
print(f'Número de linhas: {df1.shape[0]}')
print(f'Número de de colunas: {df1.shape[1]}')

Número de linhas: 1761
Número de de colunas: 35


In [76]:
#numero de valores faltantes #n\ para quebra de linha
print(f'quantidade de números faltantes:\n{df1.isna().sum():}')

#% valores faltantes
df1.isna().sum() / df1.shape[0]


quantidade de números faltantes:
Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsIn

Age                         0.0
Attrition                   0.0
BusinessTravel              0.0
DailyRate                   0.0
Department                  0.0
DistanceFromHome            0.0
Education                   0.0
EducationField              0.0
EmployeeCount               0.0
EmployeeNumber              0.0
EnvironmentSatisfaction     0.0
Gender                      0.0
HourlyRate                  0.0
JobInvolvement              0.0
JobLevel                    0.0
JobRole                     0.0
JobSatisfaction             0.0
MaritalStatus               0.0
MonthlyIncome               0.0
MonthlyRate                 0.0
NumCompaniesWorked          0.0
Over18                      0.0
OverTime                    0.0
PercentSalaryHike           0.0
PerformanceRating           0.0
RelationshipSatisfaction    0.0
StandardHours               0.0
StockOptionLevel            0.0
TotalWorkingYears           0.0
TrainingTimesLastYear       0.0
WorkLifeBalance             0.0
YearsAtC

In [77]:
#tipo dos dados
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 1761 entries, 0 to 1760
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1761 non-null   int64
 1   Attrition                 1761 non-null   str  
 2   BusinessTravel            1761 non-null   str  
 3   DailyRate                 1761 non-null   int64
 4   Department                1761 non-null   str  
 5   DistanceFromHome          1761 non-null   int64
 6   Education                 1761 non-null   int64
 7   EducationField            1761 non-null   str  
 8   EmployeeCount             1761 non-null   int64
 9   EmployeeNumber            1761 non-null   int64
 10  EnvironmentSatisfaction   1761 non-null   int64
 11  Gender                    1761 non-null   str  
 12  HourlyRate                1761 non-null   int64
 13  JobInvolvement            1761 non-null   int64
 14  JobLevel                  1761 non-null   int64
 15

# 3.0 Análise Descritiva

In [78]:
#primeira coisa vamos separar os dados em variáveis categóricas e numéricas

num_atributos = df1.select_dtypes(include=['int64', 'float64'])
cat_atributos = df1.select_dtypes(exclude=['int64', 'float64'])

In [79]:
#verificando como ficou cat_numerica
print(num_atributos.shape)
num_atributos.head(5)

(1761, 26)


,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,58,601,7,4,1,1360,3,53,2,3,...,4,80,0,31,0,2,10,9,5,9
1,27,1341,21,2,1,1950,0,58,1,0,...,1,80,0,0,2,3,0,0,0,0
2,50,407,28,3,1,2049,3,39,1,2,...,2,80,1,19,3,2,2,2,1,0
3,39,439,9,3,1,1132,3,70,3,2,...,3,80,0,9,2,2,5,4,0,3
4,32,601,7,5,1,1446,4,97,3,2,...,3,80,1,7,3,2,4,3,0,3


In [80]:
#verificanduto cutomuto ficutou cat_categutorica
print(cat_atributos.shape)
cat_atributos.head(5).T

(1761, 9)


,0,1,2,3,4
Attrition,Yes,Yes,Yes,No,No
BusinessTravel,Travel_Rarely,Travel_Frequently,Travel_Rarely,Non-Travel,Travel_Rarely
Department,Research & Development,Human Resources,Sales,Research & Development,Sales
EducationField,Medical,Human Resources,Marketing,Life Sciences,Marketing
Gender,Female,Female,Male,Male,Male
JobRole,Manufacturing Director,Human Resources,Sales Executive,Laboratory Technician,Sales Executive
MaritalStatus,Married,Married,Divorced,Single,Married
Over18,Y,Y,Y,Y,Y
OverTime,Yes,No,Yes,No,No


In [ ]:
# Atributos Numerais


#aqui vamos criar series (coluna em lista) e depois transpor para linhas 
#porque series inverte na vertical, formando colunas. Nosso interesse 
#é linhas pra juntar tudo depois cancatenando, formando um dataset com essas infos

#tendencia central: media e mediana
ct1 = pd.DataFrame(num_atributos.apply(np.mean)).T
ct2 = pd.DataFrame((num_atributos).apply(np.median)).T

#dispersão: desvio, min, max, range, skew e kurtosis
dt1 = pd.DataFrame(num_atributos.apply(np.std)).T
dt2 = pd.DataFrame(num_atributos.apply(np.min)).T
dt3 = pd.DataFrame(num_atributos.apply(np.max)).T

#nesse caso usamos lambda. nomeamos uma variavel x para fazer os calculos
dt4 = pd.DataFrame(num_atributos.apply(lambda x: x.max() - x.min())).T
dt5 = pd.DataFrame(num_atributos.apply(lambda x: x.skew())).T
dt6 = pd.DataFrame(num_atributos.apply(lambda x: x.kurtosis())).T


In [88]:
#juntando todas as linhas para criar um dataset robusto, com concat
m = pd.concat([ct1, ct2, dt1, dt2, dt3, dt4, dt5, dt6], axis=0).T.reset_index()

#renomeando as colunas de m, colocando na ordem certa 
#colocou nome para as colunas descricao como 'atributtes
m.columns = ['attributes', 'mean', 'median', 'std', 'min', 'max', 'range', 'skew', 'kurtosis']

m.style.background_gradient(cmap='Blues')

,attributes,mean,median,std,min,max,range,skew,kurtosis
0,Age,36.369108,35.000000,9.373446,17.000000,60.000000,43.000000,0.418941,-0.422710
1,DailyRate,793.485520,775.000000,403.011692,97.000000,1499.000000,1402.000000,0.046807,-1.207515
2,DistanceFromHome,9.373083,7.000000,8.298036,0.000000,29.000000,29.000000,0.875237,-0.445472
3,Education,2.819989,3.000000,1.066749,0.000000,5.000000,5.000000,-0.294275,-0.454032
4,EmployeeCount,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
5,EmployeeNumber,1030.098807,1030.000000,600.607713,0.000000,2068.000000,2068.000000,0.001871,-1.213682
6,EnvironmentSatisfaction,2.589438,3.000000,1.162582,0.000000,4.000000,4.000000,-0.335475,-1.043544
7,HourlyRate,65.768313,66.000000,20.234703,30.000000,100.000000,70.000000,-0.014268,-1.176861
8,JobInvolvement,2.600227,3.000000,0.803372,0.000000,4.000000,4.000000,-0.673451,0.550358
9,JobLevel,1.932425,2.000000,1.156901,0.000000,5.000000,5.000000,0.843239,0.391748
